In [ ]:
import sys
import os
import numpy as np
import yaml
import logging
from pathlib import Path
import time

import matplotlib.pyplot as plt
%matplotlib inline

# --- Add paths for custom modules ---
# Adjust these paths if your notebook is in a different location
# Option 1: If notebook is in repo_root/notebooks and helpers in repo_root/notebooks
# Option 2: If notebook is in repo_root/test_py and helpers in repo_root/test_py
# Option 3: If notebook is elsewhere, point to the correct locations

# Assuming the notebook is in a 'notebooks' subdirectory of the repository root
try:
    repo_root = Path.cwd().parent
    build_dir = repo_root / 'build' # For mpy_detector.so
    helpers_dir = Path.cwd() # Assuming notebook_helpers.py is in the same dir as the notebook

    if str(build_dir) not in sys.path:
        sys.path.insert(0, str(build_dir))
    if str(helpers_dir) not in sys.path:
        sys.path.insert(0, str(helpers_dir))
    log_path_added = True
except Exception as e:
    print(f"Error setting up sys.path: {e}")
    log_path_added = False

# --- Import custom and NuScenes modules ---
try:
    import mpy_detector as mdet
    if not log_path_added: # Attempt to import helpers if path setup failed but they are findable
        import notebook_helpers as nh
    else: # Normal import after path setup
        import notebook_helpers as nh

    from nuscenes.nuscenes import NuScenes
    from nuscenes.utils.data_classes import LidarPointCloud
    from pyquaternion import Quaternion
except ImportError as e:
    print(f"CRITICAL: Failed to import mpy_detector, notebook_helpers, or NuScenes components: {e}")
    print(f"Ensure mpy_detector.so is in the build directory ({build_dir if 'build_dir' in locals() else 'UNKNOWN'}) and it's in sys.path.")
    print(f"Ensure notebook_helpers.py is in {helpers_dir if 'helpers_dir' in locals() else 'UNKNOWN'} and it's in sys.path.")
    print(f"Python sys.path: {sys.path}")
    # You might want to raise an error here or skip parts of the notebook
    mdet = None
    nh = None


# --- Logging Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
log = logging.getLogger("NuScenesNotebook")
if nh: log.info("Notebook helpers imported.")
if mdet: log.info("mpy_detector imported.")
log.info("NuScenes devkit available.")

# --- Configuration ---
# Assuming notebook is in a subdir of repo_root
CONFIG_PATH_STR = str(repo_root / "test_data/configs/test_full_config_low_log.yaml")
if not Path(CONFIG_PATH_STR).exists():
    log.error(f"Config file not found: {CONFIG_PATH_STR}")
    CONFIG_PATH_STR = None # Handle this downstream

NU_DATAROOT = None
NU_VERSION = 'v1.0-mini' # or 'v1.0-trainval'
if CONFIG_PATH_STR:
    try:
        with open(CONFIG_PATH_STR, 'r') as f:
            full_config = yaml.safe_load(f)
        if full_config and 'dataset' in full_config:
            NU_DATAROOT_RAW = full_config['dataset'].get('dataroot')
            if NU_DATAROOT_RAW:
                NU_DATAROOT = str(Path(NU_DATAROOT_RAW).expanduser())
            NU_VERSION = full_config['dataset'].get('version', NU_VERSION)
        log.info(f"Using NuScenes dataroot: {NU_DATAROOT}, version: {NU_VERSION}")
    except Exception as e:
        log.error(f"Error loading dataset config from {CONFIG_PATH_STR}: {e}")
else:
    log.warning("Config path not set. NuScenes dataroot and version might be incorrect.")

# --- Parameters for this run ---
NU_SCENE_INDEX_TO_LOAD = 0  # Which scene to load from NuScenes
NU_NUM_SAMPLES_TO_PROCESS = 10 # How many consecutive lidar scans to process from the scene
# For visualization
PLOT_X_LIMS = (-50, 50) # Bird's eye view X limits
PLOT_Y_LIMS = (-50, 50) # Bird's eye view Y limits
POINT_SIZE = 2

In [ ]:
if mdet is None or CONFIG_PATH_STR is None:
    log.error("mpy_detector or config path not available. Cannot initialize filter.")
    filt = None
else:
    log.info(f"Initializing DynObjFilter with config: {CONFIG_PATH_STR}")
    try:
        filt = mdet.DynObjFilter(config_path=CONFIG_PATH_STR)
        log.info("DynObjFilter initialized successfully.")
        log.info(f"Filter initial map count: {filt.get_depth_map_count()}")
    except Exception as e:
        log.error(f"Error initializing DynObjFilter: {e}", exc_info=True)
        filt = None

nusc = None
if NU_DATAROOT and Path(NU_DATAROOT).exists():
    log.info(f"Loading NuScenes (Version: {NU_VERSION}, Dataroot: {NU_DATAROOT})...")
    try:
        nusc = NuScenes(version=NU_VERSION, dataroot=NU_DATAROOT, verbose=False)
        log.info("NuScenes loaded successfully.")
        log.info(f"Number of scenes: {len(nusc.scene)}")
    except Exception as e:
        log.error(f"Error loading NuScenes: {e}", exc_info=True)
        nusc = None
else:
    log.warning(f"NuScenes dataroot not found or not specified: {NU_DATAROOT}")

In [ ]:

sample_tokens_to_process = []
scene_name_for_run = "N/A"

if nusc and len(nusc.scene) > NU_SCENE_INDEX_TO_LOAD:
    scene_info = nusc.scene[NU_SCENE_INDEX_TO_LOAD]
    scene_name_for_run = scene_info['name']
    log.info(f"Selected Scene: {scene_name_for_run} (Index {NU_SCENE_INDEX_TO_LOAD})")

    current_sample_token = scene_info['first_sample_token']
    for _ in range(NU_NUM_SAMPLES_TO_PROCESS):
        if not current_sample_token:
            log.warning("Reached end of scene before collecting desired number of samples.")
            break
        sample_tokens_to_process.append(current_sample_token)
        sample_record = nusc.get('sample', current_sample_token)
        current_sample_token = sample_record['next']
    log.info(f"Collected {len(sample_tokens_to_process)} sample tokens for processing.")
elif nusc:
    log.error(f"Scene index {NU_SCENE_INDEX_TO_LOAD} is out of range. Max index: {len(nusc.scene)-1}")
else:
    log.error("NuScenes not loaded. Cannot select samples.")


In [ ]:
if filt and nusc and sample_tokens_to_process and nh:
    invalid_label_val = mdet.DynObjLabel.INVALID.value if hasattr(mdet, "DynObjLabel") else 0

    for seq_id, sample_token in enumerate(sample_tokens_to_process):
        log.info(f"--- Processing Sample Token: {sample_token} (Seq ID: {seq_id}) ---")
        try:
            sample_record = nusc.get('sample', sample_token)
            lidar_top_token = sample_record['data']['LIDAR_TOP']
            lidar_data = nusc.get('sample_data', lidar_top_token)
            scan_timestamp_sec = lidar_data['timestamp'] / 1e6

            # Load point cloud
            lidar_filepath_str = nusc.get_sample_data_path(lidar_top_token)
            if not Path(lidar_filepath_str).exists():
                log.warning(f"  Lidar file not found: {lidar_filepath_str}. Skipping frame {seq_id}.")
                continue
            pc = LidarPointCloud.from_file(lidar_filepath_str)
            points_sensor_frame_raw = pc.points.T # Keep as Nx4 or Nx5 initially
            # Select X, Y, Z, Intensity (ensure it's Nx4 for the filter)
            points_sensor_frame = points_sensor_frame_raw[:, :4].astype(np.float32)
            num_input_points = points_sensor_frame.shape[0]

            if num_input_points == 0:
                 log.info(f"  Frame {seq_id}: Empty point cloud. Skipping.")
                 continue
            log.info(f"  Frame {seq_id}: Loaded {num_input_points} points.")

            # Get Global Pose
            # Set debug_seq_id to a specific frame index if you want detailed pose logs from helper
            rotation_s2w, position_s2w = nh.get_sensor_global_pose(nusc, lidar_top_token, current_seq_id=seq_id, debug_seq_id=-1)

            # Call C++ Filter
            log.info(f"  Adding scan to filter (Timestamp: {scan_timestamp_sec:.6f})...")
            filter_start_time = time.time()
            filt.add_scan(
                points_np=points_sensor_frame,
                rotation=rotation_s2w,
                position=position_s2w,
                timestamp=scan_timestamp_sec
            )
            filter_end_time = time.time()
            log.info(f"  Filter processing took {filter_end_time - filter_start_time:.4f} seconds.")

            # Retrieve results for the scan we just added
            # The seq_id passed to get_processed_points_info should match the one C++ assigns.
            # If C++ uses its internal counter (last_processed_seq_id_), use that.
            # For simplicity, let's assume the current loop 'seq_id' is what we want to retrieve.
            # This might need adjustment if the filter processes asynchronously or buffers differently.
            # Let's try retrieving for the 'last_processed_seq_id' reported by the filter.
            last_processed_filter_seq_id = filt.get_last_processed_seq_id()
            log.info(f"  Attempting to retrieve processed info for filter's last_processed_seq_id: {last_processed_filter_seq_id}")

            processed_info_list = filt.get_processed_points_info(last_processed_filter_seq_id)
            log.info(f"  Filter returned info for {len(processed_info_list)} points for seq_id {last_processed_filter_seq_id}.")

            # Align filter labels to original point cloud
            filter_labels_aligned = nh.align_filter_labels(processed_info_list, num_input_points, invalid_label_val)

            # Get Ground Truth Dynamic Mask
            gt_dynamic_mask = nh.get_gt_dynamic_mask(nusc, sample_token, lidar_top_token, points_sensor_frame)
            gt_labels = np.full(num_input_points, mdet.DynObjLabel.STATIC.value if hasattr(mdet, "DynObjLabel") else 1, dtype=int) # Default to static
            gt_labels[gt_dynamic_mask] = mdet.DynObjLabel.APPEARING.value if hasattr(mdet, "DynObjLabel") else 2 # Mark GT dynamic as one of the dynamic types for coloring

            # --- Visualization ---
            plot_data = [
                {'points': points_sensor_frame, 'labels': None, 'title': f'Frame {seq_id}: Raw Input ({num_input_points} pts)'},
                {'points': points_sensor_frame, 'labels': gt_labels, 'title': f'Frame {seq_id}: GT Dynamic (Blue)'},
                {'points': points_sensor_frame, 'labels': filter_labels_aligned, 'title': f'Frame {seq_id}: Filter Output Labels'}
            ]
            nh.plot_point_clouds_2d(plot_data,
                                    title_prefix=f"Scene: {scene_name_for_run}, Frame: {seq_id}",
                                    x_lims=PLOT_X_LIMS, y_lims=PLOT_Y_LIMS, point_size=POINT_SIZE)
            plt.show() # Ensure plot is displayed

        except Exception as e:
            log.error(f"Error processing frame {seq_id} (Sample: {sample_token}): {e}", exc_info=True)
            # Decide whether to break or continue
            # break
        break
else:
    log.warning("Filter, NuScenes, samples, or helpers not available. Skipping processing loop.")

In [ ]:
# Cell 5: Interactive C++ Utility Function Calls (Corrected)

if filt and mdet and nh and nusc and 'points_sensor_frame' in locals() and 'rotation_s2w' in locals() and 'position_s2w' in locals():
    log.info("\n--- Interactive C++ Utility Testing (with Real Data) ---")

    # Use data from the LAST scan processed in Cell 4
    # points_sensor_frame: Nx4 points in sensor frame (from last iteration of Cell 4)
    # rotation_s2w, position_s2w: Sensor-to-World transform (from last iteration of Cell 4)

    if points_sensor_frame.shape[0] > 0:
        test_point_idx_in_scan = 0  # Test with the first point of the last scan
        
        # 1. Get the point in sensor frame (X, Y, Z only for transformation)
        p_sensor_xyz = points_sensor_frame[test_point_idx_in_scan, :3]

        # 2. Transform this point to global coordinates
        # p_global = R_s2w * p_sensor + T_s2w
        p_global_xyz = rotation_s2w @ p_sensor_xyz + position_s2w
        global_point_coords_for_binding = p_global_xyz.reshape(3, 1).astype(np.float64) # Ensure 3x1 and float64

        # 3. The depth_index is the original index of the point in its scan
        depth_index_for_binding = test_point_idx_in_scan # Or if points_sensor_frame was filtered, use its original index if available

        # 4. Calculate World-to-Sensor transformation
        # R_w2s = R_s2w.T
        # T_w2s = -R_s2w.T * T_s2w
        rotation_w2s_for_binding = rotation_s2w.T.astype(np.float64)
        translation_w2s_for_binding = (-rotation_s2w.T @ position_s2w).reshape(3, 1).astype(np.float64)

        log.info(f"Testing spherical_projection with point from last scan (original index: {test_point_idx_in_scan})")
        log.info(f"  Global Coords: {global_point_coords_for_binding.flatten().tolist()}")
        log.info(f"  Rotation (W2S): \n{rotation_w2s_for_binding}")
        log.info(f"  Translation (W2S): {translation_w2s_for_binding.flatten().tolist()}")
        log.info(f"  Config Path: {CONFIG_PATH_STR}")

        try:
            sph_coords_m_tuple = mdet.spherical_projection(
                global_point_coords=global_point_coords_for_binding,
                original_idx_for_point=test_point_idx_in_scan, # <--- UPDATED ARGUMENT NAME
                rotation_world_to_sensor=rotation_w2s_for_binding,
                translation_world_to_sensor=translation_w2s_for_binding,
                config_path=CONFIG_PATH_STR
            )
            # Unpack the tuple
            sph_coords_m_numpy_array = sph_coords_m_tuple[0] # This is the py::array_t<float>
            H_ind = sph_coords_m_tuple[1]
            V_ind = sph_coords_m_tuple[2]
            GridPos = sph_coords_m_tuple[3]

            # Convert the numpy array for sph_coords_m to a simple list or ensure it's handled correctly downstream
            # For direct use, it's fine. If you were expecting a list:
            sph_coords_m = list(sph_coords_m_numpy_array) # sph_coords_m is now [az, el, depth]

            log.info(f"C++ spherical_projection output:")
            log.info(f"  Spherical Coords (Az, El, Depth): {sph_coords_m}")
            log.info(f"  H_ind: {H_ind}, V_ind: {V_ind}, GridPos: {GridPos}")


        except Exception as e:
            log.error(f"Error calling spherical_projection: {e}", exc_info=True)
    else:
        log.warning("Last processed scan had no points, cannot test spherical_projection with its data.")
else:
    log.warning("Filter, mdet, helpers, NuScenes, or data from previous cells not available for interactive C++ utility tests.")

In [ ]:
# Ensure these variables are from the same context as the spherical_projection call
# global_point_coords_for_binding (3x1 np.float64)
# rotation_w2s_for_binding (3x3 np.float64)
# translation_w2s_for_binding (3x1 np.float64)

# Calculate point coordinates in sensor frame: p_sensor = R_w2s * p_global + T_w2s
point_in_sensor_frame_xyz = rotation_w2s_for_binding @ global_point_coords_for_binding + translation_w2s_for_binding
# point_in_sensor_frame_xyz should be a 3x1 array. Let's flatten for easier use.
x_s = point_in_sensor_frame_xyz[0, 0]
y_s = point_in_sensor_frame_xyz[1, 0]
z_s = point_in_sensor_frame_xyz[2, 0]

log.info(f"Point in Sensor Frame (calculated in Python): X={x_s:.3f}, Y={y_s:.3f}, Z={z_s:.3f}")

# Manual calculation of spherical coordinates
manual_depth = np.sqrt(x_s**2 + y_s**2 + z_s**2)
# Azimuth: atan2(y,x). Ensure your C++ uses the same convention (e.g. x-forward, y-left, z-up for sensor)
# Common convention: x forward, y left, z up. Azimuth is angle in x-y plane from x-axis.
manual_azimuth_rad = np.arctan2(y_s, x_s)
# Elevation: angle with xy-plane.
manual_elevation_rad = np.arctan2(z_s, np.sqrt(x_s**2 + y_s**2)) # More robust: np.arcsin(z_s / manual_depth) if manual_depth > 0

log.info(f"Manually Calculated Spherical Coords (Sensor Frame):")
log.info(f"  Depth: {manual_depth:.2f} m")
log.info(f"  Azimuth: {manual_azimuth_rad:.3f} rad ({np.rad2deg(manual_azimuth_rad):.1f} deg)")
log.info(f"  Elevation: {manual_elevation_rad:.3f} rad ({np.rad2deg(manual_elevation_rad):.1f} deg)")

log.info(f"C++ Output Spherical Coords:")
log.info(f"  Depth: {sph_coords_m[2]:.2f} m")
log.info(f"  Azimuth: {sph_coords_m[0]:.3f} rad ({np.rad2deg(sph_coords_m[0]):.1f} deg)")
log.info(f"  Elevation: {sph_coords_m[1]:.3f} rad ({np.rad2deg(sph_coords_m[1]):.1f} deg)")

In [ ]:
# New Cell: Visualize Full Point Cloud Spherical Projection
from tqdm.auto import tqdm # Import tqdm
import matplotlib.pyplot as plt
import math # For math.pi, math.radians, math.floor, math.ceil
# Ensure numpy, yaml, mdet, logging, Path are already imported

log.info("\n--- Visualizing Full Point Cloud Spherical Projection ---")

# --- 1. Get Data from a Single Scan ---
# Option A: Re-run the first iteration of Cell 4's loop to get fresh data
# Option B: Use data if it's still in scope from a previous run of Cell 4 (less reliable)

# For Option A (recommended for a clean test):
# You might want to encapsulate the first iteration of Cell 4 into a helper function
# or re-paste parts of it here, ensuring NU_NUM_SAMPLES_TO_PROCESS = 1.
# For this example, let's assume 'points_sensor_frame', 'rotation_s2w',
# 'position_s2w' are available from the *first* processed sample.

# Make sure these variables are populated from the first scan:
# points_sensor_frame: (Nx4 numpy array) Point cloud in sensor frame
# rotation_s2w: (3x3 numpy array) Sensor to World rotation
# position_s2w: (3x1 numpy array) Sensor to World translation
# CONFIG_PATH_STR: Path to your YAML config

# Check if necessary data is available (add error handling as needed)
if 'points_sensor_frame' not in locals() or \
   'rotation_s2w' not in locals() or \
   'position_s2w' not in locals() or \
   CONFIG_PATH_STR is None:
    log.error("Required data (point cloud, pose, config path) not available. Please run Cell 4 for one sample first.")
    # raise RuntimeError("Required data not available") # Or skip this cell
else:
    log.info(f"Using point cloud with {points_sensor_frame.shape[0]} points for projection.")

    # --- 2. Calculate World-to-Sensor Transformation ---
    rotation_w2s = rotation_s2w.T.astype(np.float64)
    translation_w2s = (-rotation_s2w.T @ position_s2w).reshape(3, 1).astype(np.float64)

    # --- 3. Determine Projection Image Dimensions from Config ---
    # This is crucial for creating the image array correctly.
    # We'll parse the config file to get base parameters and calculate dimensions
    # similar to how it's done in the C++ `load_config`.

    try:
        with open(CONFIG_PATH_STR, 'r') as f:
            yaml_config = yaml.safe_load(f)
        # Assuming params are under 'dyn_obj' or at the root if 'dyn_obj' is the top key
        filter_params_yaml = yaml_config.get('dyn_obj', yaml_config)

        hor_res_rad = filter_params_yaml.get('hor_resolution_max', 0.007)
        ver_res_rad = filter_params_yaml.get('ver_resolution_max', 0.022)
        fov_up_deg = filter_params_yaml.get('fov_up', 10.0)
        fov_down_deg = filter_params_yaml.get('fov_down', -30.0)
        fov_left_deg = filter_params_yaml.get('fov_left', 180.0) # Full 360 if -180 to 180
        fov_right_deg = filter_params_yaml.get('fov_right', -180.0)

        fov_up_rad = math.radians(fov_up_deg)
        fov_down_rad = math.radians(fov_down_deg)
        fov_left_rad = math.radians(fov_left_deg)
        fov_right_rad = math.radians(fov_right_deg)

        # Calculate image dimensions based on C++ derived parameter logic
        # Vertical dimension (rows)
        # V_raw_idx = floor((elevation_rad + PI/2) / ver_res_rad)
        # V_idx_final = V_raw_idx - pixel_fov_down
        # pixel_fov_down is the V_raw_idx for fov_down_rad
        # pixel_fov_up is the V_raw_idx for fov_up_rad
        cpp_pixel_fov_up = math.floor((fov_up_rad + 0.5 * math.pi) / ver_res_rad)
        cpp_pixel_fov_down = math.floor((fov_down_rad + 0.5 * math.pi) / ver_res_rad)
        # The number of rows is the count of distinct V_raw_idx values from pixel_fov_down to pixel_fov_up
        image_height = int(cpp_pixel_fov_up - cpp_pixel_fov_down + 1)

        # Horizontal dimension (cols)
        # H_raw_idx = floor((azimuth_rad + PI) / hor_res_rad)
        # H_idx_final = H_raw_idx - pixel_fov_right (if pixel_fov_right is the 0th index of the image)
        # pixel_fov_right is H_raw_idx for fov_right_rad
        # pixel_fov_left is H_raw_idx for fov_left_rad
        cpp_pixel_fov_left = math.floor((fov_left_rad + math.pi) / hor_res_rad)
        cpp_pixel_fov_right = math.floor((fov_right_rad + math.pi) / hor_res_rad)
        image_width = int(cpp_pixel_fov_left - cpp_pixel_fov_right + 1)

        # Handle potential wrap-around for 360-degree horizontal FOV
        # If fov_left is 180 and fov_right is -180, it implies a full 360 sweep.
        # In this case, cpp_pixel_fov_right would be 0.
        # And cpp_pixel_fov_left would be floor(2*pi / hor_res_rad).
        # So image_width = floor(2*pi / hor_res_rad) + 1.
        # A common convention is that the number of bins is ceil(range_angle / resolution).
        # For 360 deg: image_width = ceil(2*pi / hor_res_rad)
        if abs(fov_left_deg - fov_right_deg) >= 359.9: # Check for full horizontal FOV
             image_width = int(math.ceil(2 * math.pi / hor_res_rad))


        log.info(f"Calculated projection image dimensions: Width={image_width}, Height={image_height}")
        if image_width <= 0 or image_height <= 0:
            raise ValueError("Calculated image dimensions are invalid. Check config parameters.")

    except Exception as e:
        log.error(f"Error calculating image dimensions from config: {e}. Using fallback or aborting.", exc_info=True)
        # You might want to set fallback dimensions or re-raise
        image_width, image_height = 64, 1024 # Example fallback, adjust as needed
        log.warning(f"Using fallback image dimensions: Width={image_width}, Height={image_height}")
        # raise e

    
    # --- 4. Initialize Depth Image ---
    # Use np.nan for pixels with no data, so they are not plotted or are plotted with a specific color.
    # Or use a large number (e.g., max_range + 1) if your colormap handles it.
    depth_image = np.full((image_height, image_width), np.nan, dtype=np.float32)
    # To store count of points per pixel (optional, for debugging density)
    # count_image = np.zeros((image_height, image_width), dtype=np.int32)

    # --- 5. Project All Points and Populate Depth Image ---
    projected_points_count = 0
    num_total_points = points_sensor_frame.shape[0]

    # Wrap the range iterator with tqdm for a progress bar
    for i in tqdm(range(num_total_points), desc="Projecting Points", unit="point"):
        p_sensor_xyz = points_sensor_frame[i, :3] # X, Y, Z

        # Transform point to global coordinates
        p_global_xyz = (rotation_s2w @ p_sensor_xyz + position_s2w).reshape(3, 1).astype(np.float64)

        try:
            sph_tuple = mdet.spherical_projection(
                global_point_coords=p_global_xyz,
                original_idx_for_point=i, # Original index in this scan
                rotation_world_to_sensor=rotation_w2s,
                translation_world_to_sensor=translation_w2s,
                config_path=CONFIG_PATH_STR
            )
            sph_coords_sensor = sph_tuple[0] # [az_sensor, el_sensor, depth_sensor]
            h_ind = sph_tuple[1] # Column
            v_ind = sph_tuple[2] # Row
            grid_pos_flag = sph_tuple[3] # Validity flag

            # Check if projection is valid and within calculated image bounds
            if grid_pos_flag > 0: # Assuming positive flag means valid
                # Ensure indices are within the bounds of our depth_image
                if 0 <= v_ind < image_height and 0 <= h_ind < image_width:
                    depth_val = sph_coords_sensor[2] # depth_sensor

                    # Store the closest point's depth
                    if np.isnan(depth_image[v_ind, h_ind]) or depth_val < depth_image[v_ind, h_ind]:
                        depth_image[v_ind, h_ind] = depth_val
                    # count_image[v_ind, h_ind] += 1 # Optional
                    projected_points_count +=1
                # else: # Log if C++ returns indices outside our calculated bounds (should ideally not happen)
                #     log.debug(f"Point {i} projected to H_ind={h_ind}, V_ind={v_ind} which is outside image dims ({image_width}x{image_height}). GridPosFlag={grid_pos_flag}")

        except Exception as e:
            log.error(f"Error projecting point {i}: {e}", exc_info=False) # Set exc_info=True for full trace
            if i < 10: log.debug(f"Details for point {i}: Global={p_global_xyz.flatten().tolist()}") # Log details for first few errors


    log.info(f"Projected {projected_points_count} valid points onto the depth image.")

    # --- 6. Display the Depth Image ---
    if projected_points_count > 0:
        plt.figure(figsize=(12, max(6, image_height / (image_width/10) ) )) # Adjust figsize
        
        # Handle NaNs in colormap: typically, imshow with default colormap handles NaNs by not coloring them
        # or you can set a specific bad color: cmap.set_bad(color='black')
        current_cmap = plt.cm.get_cmap('viridis_r') # _r reverses the colormap (e.g. viridis_r: closer=yellow, further=blue)
        current_cmap.set_bad(color='black') # Color for NaN pixels

        plt.imshow(depth_image, cmap=current_cmap, aspect='auto') # 'auto' or 'equal' or set aspect ratio
        plt.colorbar(label='Depth (m)')
        plt.xlabel("Horizontal Index (u / Azimuth bin)")
        plt.ylabel("Vertical Index (v / Elevation bin)")
        plt.title(f"Spherical Projection Depth Map (Scan from Seq ID {seq_id if 'seq_id' in locals() else 'N/A'})")
        plt.tight_layout()
        plt.show()
    else:
        log.warning("No points were successfully projected to the depth image. Cannot display.")

# End of new cell

In [ ]:
# New Cell: Visualize Full Point Cloud Spherical Projection (MANUAL Python Calculation with tqdm)

import matplotlib.pyplot as plt
import math # For math.pi, math.radians, math.floor, math.ceil, atan2, asin, sqrt
from tqdm.auto import tqdm # Import tqdm
import numpy as np

# Ensure yaml, mdet, logging, Path are already imported

log.info("\n--- Visualizing Full Point Cloud Spherical Projection (MANUAL Python Calculation with tqdm) ---")

# --- 1. Get Data from a Single Scan ---
# (Assuming 'points_sensor_frame', 'rotation_s2w', 'position_s2w', 'CONFIG_PATH_STR'
# and 'seq_id' (if used in title) are available from the first processed sample of Cell 4)

if 'points_sensor_frame' not in locals() or \
   'rotation_s2w' not in locals() or \
   'position_s2w' not in locals() or \
   CONFIG_PATH_STR is None:
    log.error("Required data (point cloud, pose, config path) not available. Please run Cell 4 for one sample first.")
    # raise RuntimeError("Required data not available") # Or skip this cell
else:
    log.info(f"Using point cloud with {points_sensor_frame.shape[0]} points for manual projection.")

    # --- 2. Load Parameters from Config (needed for manual projection logic) ---
    try:
        with open(CONFIG_PATH_STR, 'r') as f:
            yaml_config = yaml.safe_load(f)
        filter_params_yaml = yaml_config.get('dyn_obj', yaml_config)

        hor_res_rad = filter_params_yaml.get('hor_resolution_max', 0.007)
        ver_res_rad = filter_params_yaml.get('ver_resolution_max', 0.022)
        fov_up_deg = filter_params_yaml.get('fov_up', 10.0)
        fov_down_deg = filter_params_yaml.get('fov_down', -30.0)
        fov_left_deg = filter_params_yaml.get('fov_left', 180.0) # Used for FOV check if not full 360
        fov_right_deg = filter_params_yaml.get('fov_right', -180.0)# Used for FOV check if not full 360
        blind_dis = filter_params_yaml.get('blind_dis', 0.2)

        fov_up_rad = math.radians(fov_up_deg)
        fov_down_rad = math.radians(fov_down_deg)
        # These are for strict FOV checks if needed, atan2 covers [-pi, pi]
        # fov_left_rad_strict = math.radians(fov_left_deg)
        # fov_right_rad_strict = math.radians(fov_right_deg)


        # --- 3. Determine Projection Image Dimensions (same as before) ---
        # This logic should be identical to the previous cell to ensure consistency
        cpp_pixel_fov_up_abs = math.floor((fov_up_rad + 0.5 * math.pi) / ver_res_rad)
        cpp_pixel_fov_down_abs = math.floor((fov_down_rad + 0.5 * math.pi) / ver_res_rad)
        image_height = int(cpp_pixel_fov_up_abs - cpp_pixel_fov_down_abs + 1)

        is_full_horizontal_fov = abs(fov_left_deg - fov_right_deg) >= 359.9
        if is_full_horizontal_fov:
             image_width = int(math.ceil(2 * math.pi / hor_res_rad))
        else:
            log.warning("Manual projection for partial horizontal FOV needs careful index mapping. Assuming full 360 for width calculation.")
            # For non-360, the C++ mapping of fov_left/right to pixel indices is critical.
            # Here, we'll proceed with a full 360 width, and points outside a conceptual
            # fov_left/right range (if stricter than -pi to pi) would be filtered by angle checks.
            image_width = int(math.ceil(2 * math.pi / hor_res_rad))


        log.info(f"Calculated projection image dimensions (manual): Width={image_width}, Height={image_height}")
        if image_width <= 0 or image_height <= 0:
            raise ValueError("Calculated image dimensions are invalid (manual). Check config parameters.")

    except Exception as e:
        log.error(f"Error loading parameters or calculating image dimensions (manual): {e}. Using fallback or aborting.", exc_info=True)
        image_width, image_height = 64, int(2 * math.pi / 0.007) # Example fallback
        log.warning(f"Using fallback image dimensions (manual): Width={image_width}, Height={image_height}")


    # --- 4. Initialize Depth Image for Manual Projection ---
    depth_image_manual = np.full((image_height, image_width), np.nan, dtype=np.float32)

    # --- 5. Project All Points Manually and Populate Depth Image ---
    manual_projected_points_count = 0
    num_total_points = points_sensor_frame.shape[0]

    # Pre-calculate PI and PI/2 for the loop
    PI = math.pi
    HALF_PI = PI / 2.0

    for i in tqdm(range(num_total_points), desc="Manually Projecting Points", unit="point"):
        # Points are already in sensor frame (points_sensor_frame is Nx4: x,y,z,intensity)
        x_s = points_sensor_frame[i, 0]
        y_s = points_sensor_frame[i, 1]
        z_s = points_sensor_frame[i, 2]

        # a. Calculate depth (range) in sensor frame
        depth_sensor = math.sqrt(x_s**2 + y_s**2 + z_s**2)

        # b. Check blind distance
        if depth_sensor < blind_dis:
            continue

        # c. Calculate azimuth and elevation in sensor frame
        # Azimuth: atan2(y, x) -> [-PI, PI]
        azimuth_sensor_rad = math.atan2(y_s, x_s)
        # Elevation: asin(z / depth) -> [-PI/2, PI/2]
        if depth_sensor == 0: # Should be caught by blind_dis, but as a safeguard
            elevation_sensor_rad = 0
        else:
            elevation_sensor_rad = math.asin(z_s / depth_sensor)

        # d. Check if angles are within sensor's FOV
        # Vertical FOV check
        if not (fov_down_rad <= elevation_sensor_rad <= fov_up_rad):
            continue
        # Horizontal FOV check (only relevant if not full 360, but atan2 range is [-PI,PI])
        # If you had a stricter fov_left/right_rad_strict for a non-360 sensor:
        # if not (fov_right_rad_strict <= azimuth_sensor_rad <= fov_left_rad_strict): # This logic is tricky for wrap-around
        #    continue                                                               # For now, assume full 360 is fine

        # e. Convert azimuth and elevation to grid indices (h_ind, v_ind)
        # Vertical index (v_ind or row)
        # v_raw_abs is the index if the grid started at -PI/2 elevation
        v_raw_abs = math.floor((elevation_sensor_rad + HALF_PI) / ver_res_rad)
        # v_ind is 0-indexed relative to the start of our FOV (fov_down_rad)
        # cpp_pixel_fov_down_abs was calculated earlier
        v_ind = int(v_raw_abs - cpp_pixel_fov_down_abs)

        # Horizontal index (h_ind or column)
        # h_raw_abs is the index if the grid started at -PI azimuth
        h_raw_abs = math.floor((azimuth_sensor_rad + PI) / hor_res_rad)
        # For full 360, h_ind is typically h_raw_abs if the image starts at -PI azimuth.
        # If fov_right_deg was used to define the 0th column, you'd subtract its corresponding raw index.
        # Assuming image_width covers the full 2*PI range starting from -PI:
        h_ind = int(h_raw_abs)


        # f. Populate depth image
        if 0 <= v_ind < image_height and 0 <= h_ind < image_width:
            if np.isnan(depth_image_manual[v_ind, h_ind]) or depth_sensor < depth_image_manual[v_ind, h_ind]:
                depth_image_manual[v_ind, h_ind] = depth_sensor
            manual_projected_points_count += 1
        # else: # Debug if calculated indices fall out of expected range
            # if i < 100 and i % 10 == 0 : # Log sparsely for debugging
            #     log.debug(f"Man.Proj Point {i}: az={math.degrees(azimuth_sensor_rad):.1f}, el={math.degrees(elevation_sensor_rad):.1f} -> H={h_ind}, V={v_ind} (rawV={v_raw_abs}, rawH={h_raw_abs})")


    log.info(f"Manually projected {manual_projected_points_count} valid points onto the depth image.")

    # --- 6. Display the Manually Calculated Depth Image ---
    if manual_projected_points_count > 0:
        plt.figure(figsize=(12, max(6, image_height / (image_width/10) if image_width > 0 else 6 )))
        current_cmap = plt.cm.get_cmap('viridis_r') # Or 'jet', 'plasma_r', etc.
        current_cmap.set_bad(color='black') # Color for NaN pixels

        plt.imshow(depth_image_manual, cmap=current_cmap, aspect='auto')
        plt.colorbar(label='Depth (m)')
        plt.xlabel("Horizontal Index (u / Azimuth bin) - Manual")
        plt.ylabel("Vertical Index (v / Elevation bin) - Manual")
        current_seq_id_for_title = locals().get('seq_id', 'N/A')
        plt.title(f"MANUAL Spherical Projection Depth Map (Scan from Seq ID {current_seq_id_for_title})")
        plt.tight_layout()
        plt.show()
    else:
        log.warning("No points were successfully projected to the manual depth image. Cannot display.")

# End of new cell

In [ ]:
# In the cell that calls mdet.spherical_projection (the C++ one)
# Add more detailed comparison with the manual Python calculation logic

log.info("\n--- Detailed C++ vs. Python Projection Comparison ---")

# --- Ensure all necessary parameters for manual calculation are loaded here ---
# (Copy from the manual projection cell: hor_res_rad, ver_res_rad, fov_up_rad, fov_down_rad,
#  blind_dis, PI, HALF_PI, cpp_pixel_fov_down_abs, image_width_py, image_height_py)
# Note: Use distinct names for python-calculated image_width/height if they might differ
# from what C++ might be internally using. For now, assume they are intended to be the same.

try:
    with open(CONFIG_PATH_STR, 'r') as f:
        yaml_config_comp = yaml.safe_load(f)
    filter_params_yaml_comp = yaml_config_comp.get('dyn_obj', yaml_config_comp)

    hor_res_rad_comp = filter_params_yaml_comp.get('hor_resolution_max')
    ver_res_rad_comp = filter_params_yaml_comp.get('ver_resolution_max')
    fov_up_deg_comp = filter_params_yaml_comp.get('fov_up')
    fov_down_deg_comp = filter_params_yaml_comp.get('fov_down')
    # fov_left_deg_comp = filter_params_yaml_comp.get('fov_left') # For more advanced H_ind calc
    # fov_right_deg_comp = filter_params_yaml_comp.get('fov_right')# For more advanced H_ind calc
    blind_dis_comp = filter_params_yaml_comp.get('blind_dis')

    fov_up_rad_comp = math.radians(fov_up_deg_comp)
    fov_down_rad_comp = math.radians(fov_down_deg_comp)

    PI_comp = math.pi
    HALF_PI_comp = PI_comp / 2.0

    # Python's calculation of absolute pixel indices (as in manual cell)
    py_cpp_pixel_fov_up_abs = math.floor((fov_up_rad_comp + HALF_PI_comp) / ver_res_rad_comp)
    py_cpp_pixel_fov_down_abs = math.floor((fov_down_rad_comp + HALF_PI_comp) / ver_res_rad_comp)
    # Python's calculation of image dimensions (as in manual cell)
    image_height_for_py_check = int(py_cpp_pixel_fov_up_abs - py_cpp_pixel_fov_down_abs + 1)
    image_width_for_py_check = int(math.ceil(2 * PI_comp / hor_res_rad_comp)) # Assuming full 360 for Python check

except Exception as e:
    log.error(f"Error loading comparison parameters: {e}")
    # Abort or use defaults if critical params missing

cpp_valid_count = 0
py_would_be_valid_count = 0
discrepancy_count = 0
max_discrepancies_to_log = 20

# (rotation_w2s and translation_w2s should be defined from sensor pose)

for i in tqdm(range(points_sensor_frame.shape[0]), desc="C++ Proj. w/ Py Detail Comp.", unit="pt"):
    p_sensor_xyz = points_sensor_frame[i, :3]
    p_global_for_cpp = (rotation_s2w @ p_sensor_xyz + position_s2w).reshape(3, 1).astype(np.float64)

    # --- C++ Call ---
    cpp_grid_pos = 0
    cpp_h_ind, cpp_v_ind = -1, -1
    cpp_depth_val = np.nan
    cpp_az_sensor, cpp_el_sensor = np.nan, np.nan # If C++ returns these

    try:
        sph_tuple = mdet.spherical_projection(
            global_point_coords=p_global_for_cpp,
            original_idx_for_point=i,
            rotation_world_to_sensor=rotation_w2s,
            translation_world_to_sensor=translation_w2s,
            config_path=CONFIG_PATH_STR
        )
        cpp_sph_coords_arr = sph_tuple[0] # Expect [az_sensor, el_sensor, depth_sensor]
        cpp_h_ind = sph_tuple[1]
        cpp_v_ind = sph_tuple[2]
        cpp_grid_pos = sph_tuple[3]

        if cpp_grid_pos > 0:
            cpp_valid_count += 1
            cpp_az_sensor = cpp_sph_coords_arr[0]
            cpp_el_sensor = cpp_sph_coords_arr[1]
            cpp_depth_val = cpp_sph_coords_arr[2]

    except Exception as e_cpp:
        if discrepancy_count < max_discrepancies_to_log : # Log error for C++ call
            log.error(f"Point {i}: C++ mdet.spherical_projection call failed: {e_cpp}")
        # cpp_grid_pos remains 0 or some error indicator
        pass


    # --- Python Manual Calculation for the SAME sensor point p_sensor_xyz ---
    py_is_valid_this_point = False
    py_h_ind, py_v_ind = -1, -1
    py_depth_val = np.nan
    py_az_rad, py_el_rad = np.nan, np.nan

    x_s, y_s, z_s = p_sensor_xyz[0], p_sensor_xyz[1], p_sensor_xyz[2]
    current_py_depth = math.sqrt(x_s**2 + y_s**2 + z_s**2)

    if current_py_depth >= blind_dis_comp:
        py_az_rad = math.atan2(y_s, x_s)
        py_el_rad = math.asin(z_s / current_py_depth) if current_py_depth > 0 else 0

        if (fov_down_rad_comp <= py_el_rad <= fov_up_rad_comp):
            # Python's index calculation (from manual cell)
            v_raw_abs_py = math.floor((py_el_rad + HALF_PI_comp) / ver_res_rad_comp)
            py_v_ind = int(v_raw_abs_py - py_cpp_pixel_fov_down_abs)

            h_raw_abs_py = math.floor((py_az_rad + PI_comp) / hor_res_rad_comp)
            py_h_ind = int(h_raw_abs_py) # Assumes 0th col at -PI azimuth

            if 0 <= py_v_ind < image_height_for_py_check and 0 <= py_h_ind < image_width_for_py_check:
                py_is_valid_this_point = True
                py_depth_val = current_py_depth
                if cpp_grid_pos <= 0 : # Only count if C++ rejected it
                    py_would_be_valid_count +=1


    # --- Log Discrepancies ---
    if py_is_valid_this_point and cpp_grid_pos <= 0:
        discrepancy_count += 1
        if discrepancy_count <= max_discrepancies_to_log:
            log.warning(f"DISCREPANCY Point Original Index {i}: Python VALID, C++ INVALID")
            log.info(f"  Sensor XYZ: [{x_s:.3f}, {y_s:.3f}, {z_s:.3f}]")
            log.info(f"  Python Calc: Az={math.degrees(py_az_rad):.2f} El={math.degrees(py_el_rad):.2f} D={py_depth_val:.3f}")
            log.info(f"               H_ind={py_h_ind} V_ind={py_v_ind}")
            log.info(f"  C++ Output : grid_pos={cpp_grid_pos}, H_ind={cpp_h_ind}, V_ind={cpp_v_ind}")
            if not np.isnan(cpp_az_sensor): # If C++ returned sph_coords even if invalid grid_pos
                 log.info(f"               (C++ Sph Coords if avail: Az={math.degrees(cpp_az_sensor):.2f} El={math.degrees(cpp_el_sensor):.2f} D={cpp_depth_val:.3f})")
            # What are the C++ H_ind, V_ind values when grid_pos is invalid? Are they -1? Or out of bounds?
            # This comparison is key.

log.info(f"Total points processed: {points_sensor_frame.shape[0]}")
log.info(f"C++ valid projections: {cpp_valid_count}")
log.info(f"Points Python would consider valid but C++ invalidated: {py_would_be_valid_count}")
log.info(f"Total discrepancies logged in detail: {min(discrepancy_count, max_discrepancies_to_log)}")

# Then proceed to create the C++ depth image using only cpp_valid_count points
# ... (your existing imshow code for the C++ result) ...